# Module 12 — Anomaly Detection with Isolation Forest

> **Track 2 · Revolut Fintech Specialization**  
> **Difficulty**: Intermediate → Advanced  
> **Runtime**: ~5 minutes  
> **Key Libraries**: Scikit-learn, NumPy, Pandas, Plotly, Seaborn  
> **Prerequisite**: Module 11 (Rule-Based Fraud)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Transaction Dataset with 30 Features](#3-synthetic-transaction-dataset)
4. [Exploratory Data Analysis](#4-exploratory-data-analysis)
5. [Isolation Forest: Theory & Training](#5-isolation-forest-theory--training)
6. [Anomaly Score Distribution](#6-anomaly-score-distribution)
7. [Contamination Parameter Tuning](#7-contamination-parameter-tuning)
8. [t-SNE Visualization in 2D](#8-t-sne-visualization-in-2d)
9. [Comparison with Labeled Data](#9-comparison-with-labeled-data)
10. [Feature Importance (Which Features Drive Anomalies?)](#10-feature-importance)
11. [Visualization Dashboard](#11-visualization-dashboard)
12. [Key Business Takeaway](#12-key-business-takeaway)

---
## §1 · Business Context

### Why Isolation Forest at Revolut?

Rule-based systems (Module 11) catch **known fraud patterns**, but fraudsters constantly evolve.
Revolut needs an **unsupervised** method that flags **novel anomalies** without labeled data.

**Isolation Forest** is ideal because:
- **Unsupervised**: no labeled fraud data needed (critical for new fraud types).
- **Fast**: trains in O(n log n) and scores in O(1) per sample.
- **Scalable**: handles millions of transactions with 30+ features.
- **Interpretable**: anomaly scores can be thresholded and explained.

| Use Case | Team | Impact |
|----------|------|--------|
| **Novel fraud detection** | Fraud Ops | Catch patterns rules miss |
| **Account takeover** | Security | Flag unusual login/transaction combos |
| **Money laundering** | Compliance | Detect structured transactions |
| **Model monitoring** | Data Science | Flag drift in feature distributions |

In this notebook we will:
1. Generate a 200K-transaction dataset with **30 engineered features**.
2. Train an Isolation Forest and interpret anomaly scores.
3. Tune the **contamination parameter** against labeled fraud.
4. Visualize anomalies in 2D using **t-SNE**.
5. Identify which features drive anomalies (feature importance).

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import time
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, precision_recall_curve
)

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 35)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Transaction Dataset with 30 Features

We simulate **200,000 transactions** with **30 engineered features** that mimic
Revolut's fraud-scoring feature store.

In [ ]:
# ── Configuration ────────────────────────────────────────────────
N_TXN = 200_000
N_FRAUD = 4_000  # 2% fraud rate
N_LEGIT = N_TXN - N_FRAUD
N_FEATURES = 30

print(f"Generating {N_TXN:,} transactions with {N_FEATURES} features …")
print(f"  Fraud rate: {N_FRAUD / N_TXN * 100:.1f}%")

np.random.seed(42)

# ── Feature names (mimicking Revolut's feature store) ────────────
feature_names = [
    "amount",                     # 0: Transaction amount
    "amount_log",                 # 1: Log-transformed amount
    "hour_of_day",                # 2: Hour (0-23)
    "day_of_week",                # 3: Day (0=Mon, 6=Sun)
    "is_weekend",                 # 4: Weekend flag
    "txn_count_1h",               # 5: Txns in last 1 hour
    "txn_count_24h",              # 6: Txns in last 24 hours
    "txn_count_7d",               # 7: Txns in last 7 days
    "amount_sum_1h",              # 8: Total amount last 1 hour
    "amount_sum_24h",             # 9: Total amount last 24 hours
    "amount_sum_7d",              # 10: Total amount last 7 days
    "amount_mean_7d",             # 11: Mean amount last 7 days
    "amount_std_7d",              # 12: Std amount last 7 days
    "amount_max_7d",              # 13: Max amount last 7 days
    "days_since_last_txn",        # 14: Days since previous transaction
    "merchant_risk_score",        # 15: Merchant risk (0-100)
    "country_risk_score",         # 16: Country risk (0-100)
    "is_international",           # 17: International flag
    "distance_from_home_km",      # 18: Distance from home
    "velocity_1h",                # 19: Amount per hour (rolling)
    "velocity_24h",               # 20: Amount per day (rolling)
    "unique_merchants_7d",        # 21: Unique merchants last 7 days
    "unique_countries_7d",        # 22: Unique countries last 7 days
    "device_mobile",              # 23: Mobile device flag
    "device_card",                # 24: Card device flag
    "is_atm",                     # 25: ATM withdrawal flag
    "is_transfer",                # 26: Transfer flag
    "user_tenure_days",           # 27: Account age
    "credit_score",               # 28: User credit score
    "z_score_amount",             # 29: Z-score vs user's history
]

# ── Generate legitimate transactions ─────────────────────────────
X_legit = np.random.randn(N_LEGIT, N_FEATURES)

# Make features realistic
X_legit[:, 0] = np.abs(np.random.lognormal(3.0, 1.0, N_LEGIT))  # amount
X_legit[:, 1] = np.log1p(X_legit[:, 0])                          # amount_log
X_legit[:, 2] = np.random.randint(0, 24, N_LEGIT)                # hour
X_legit[:, 3] = np.random.randint(0, 7, N_LEGIT)                 # day_of_week
X_legit[:, 4] = (X_legit[:, 3] >= 5).astype(float)               # is_weekend
X_legit[:, 5] = np.random.poisson(1, N_LEGIT)                    # txn_count_1h
X_legit[:, 6] = np.random.poisson(5, N_LEGIT)                    # txn_count_24h
X_legit[:, 7] = np.random.poisson(20, N_LEGIT)                   # txn_count_7d
X_legit[:, 8] = X_legit[:, 5] * X_legit[:, 0]                    # amount_sum_1h
X_legit[:, 9] = X_legit[:, 6] * X_legit[:, 0]                    # amount_sum_24h
X_legit[:, 15] = np.random.uniform(0, 50, N_LEGIT)               # merchant_risk
X_legit[:, 16] = np.random.uniform(0, 30, N_LEGIT)               # country_risk
X_legit[:, 17] = np.random.binomial(1, 0.15, N_LEGIT)            # is_international
X_legit[:, 18] = np.random.exponential(50, N_LEGIT)              # distance_from_home
X_legit[:, 27] = np.random.exponential(365, N_LEGIT)             # user_tenure
X_legit[:, 28] = np.random.normal(650, 80, N_LEGIT)              # credit_score
X_legit[:, 29] = np.random.normal(0, 1, N_LEGIT)                 # z_score

# ── Generate fraudulent transactions (anomalous) ─────────────────
X_fraud = np.random.randn(N_FRAUD, N_FEATURES)

# Fraud has different distributions (higher amounts, riskier features)
X_fraud[:, 0] = np.abs(np.random.lognormal(5.0, 1.5, N_FRAUD))   # higher amounts
X_fraud[:, 1] = np.log1p(X_fraud[:, 0])
X_fraud[:, 2] = np.random.choice([0, 1, 2, 3, 22, 23], N_FRAUD)  # late night
X_fraud[:, 5] = np.random.poisson(5, N_FRAUD)                    # high velocity
X_fraud[:, 6] = np.random.poisson(15, N_FRAUD)                   # high 24h count
X_fraud[:, 8] = X_fraud[:, 5] * X_fraud[:, 0]                    # high 1h sum
X_fraud[:, 15] = np.random.uniform(60, 100, N_FRAUD)             # high merchant risk
X_fraud[:, 16] = np.random.uniform(70, 100, N_FRAUD)             # high country risk
X_fraud[:, 17] = np.random.binomial(1, 0.60, N_FRAUD)            # mostly international
X_fraud[:, 18] = np.random.exponential(500, N_FRAUD)             # far from home
X_fraud[:, 25] = np.random.binomial(1, 0.40, N_FRAUD)            # ATM withdrawals
X_fraud[:, 26] = np.random.binomial(1, 0.30, N_FRAUD)            # transfers
X_fraud[:, 29] = np.random.normal(3, 1.5, N_FRAUD)               # high z-score

# ── Combine ──────────────────────────────────────────────────────
X = np.vstack([X_legit, X_fraud])
y = np.concatenate([np.zeros(N_LEGIT), np.ones(N_FRAUD)])

# Shuffle
idx = np.random.permutation(N_TXN)
X = X[idx]
y = y[idx]

df = pd.DataFrame(X, columns=feature_names)
df["is_fraud"] = y

print(f"✓ Shape: {df.shape}")
print(f"  Fraud: {int(y.sum()):,} ({y.mean()*100:.2f}%)")
df.head()

---
## §4 · Exploratory Data Analysis

In [ ]:
# Summary statistics for key features
key_features = ["amount", "txn_count_1h", "merchant_risk_score",
                "country_risk_score", "distance_from_home_km", "z_score_amount"]

summary = df.groupby("is_fraud")[key_features].agg(["mean", "median", "std"]).round(2)
print("Feature Statistics by Fraud Status:")
print(summary)

In [ ]:
# Correlation heatmap
fig = px.imshow(
    df[feature_names[:15]].corr(),
    color_continuous_scale="RdBu_r",
    title="Correlation Matrix (Top 15 Features)",
    aspect="auto",
)
fig.update_layout(height=500)
fig.show()

---
## §5 · Isolation Forest: Theory & Training

### How Isolation Forest Works

1. **Randomly select a feature** and a **split value** between min and max.
2. **Recursively split** until each point is isolated or max depth is reached.
3. **Anomalies are isolated faster** (fewer splits) because they are "few and different".
4. **Anomaly score** = function of average path length across all trees.

```
Normal point:   ████████████████████ (many splits to isolate → long path → low score)
Anomaly:        ███ (few splits to isolate → short path → high score)
```

In [ ]:
# ── Scale features ───────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_names])

# ── Train Isolation Forest ───────────────────────────────────────
# contamination = expected proportion of anomalies in the dataset
contamination = 0.02  # we know fraud rate is ~2%

iso_forest = IsolationForest(
    n_estimators=100,        # number of trees
    max_samples="auto",      # subsample size (default: min(256, n_samples))
    contamination=contamination,
    max_features=1.0,        # use all features
    bootstrap=False,
    random_state=42,
    n_jobs=-1,
)

print(f"Training Isolation Forest:")
print(f"  n_estimators  : 100")
print(f"  contamination : {contamination}")
print(f"  Samples       : {len(X_scaled):,}")
print(f"  Features      : {len(feature_names)}")

t0 = time.time()
iso_forest.fit(X_scaled)
train_time = time.time() - t0

print(f"\n✓ Trained in {train_time:.2f}s")

# ── Score all transactions ───────────────────────────────────────
# decision_function: higher = more normal, lower = more anomalous
anomaly_scores = iso_forest.decision_function(X_scaled)

# predict: -1 = anomaly, 1 = normal
predictions = iso_forest.predict(X_scaled)
is_anomaly = (predictions == -1).astype(int)

df["anomaly_score"] = anomaly_scores
df["predicted_anomaly"] = is_anomaly

print(f"\nPredictions:")
print(f"  Normal (1)  : {(predictions == 1).sum():,}")
print(f"  Anomaly (-1): {(predictions == -1).sum():,} ({(predictions == -1).mean()*100:.2f}%)")

---
## §6 · Anomaly Score Distribution

In [ ]:
# Distribution of anomaly scores by fraud status
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=df[df["is_fraud"] == 0]["anomaly_score"],
    name="Legitimate",
    nbinsx=100,
    marker_color="#00CC96",
    opacity=0.7,
))

fig.add_trace(go.Histogram(
    x=df[df["is_fraud"] == 1]["anomaly_score"],
    name="Fraud",
    nbinsx=100,
    marker_color="#EF553B",
    opacity=0.7,
))

fig.add_vline(x=0, line_dash="dash", line_color="black",
              annotation_text="Decision Boundary (0)")

fig.update_layout(
    title="Anomaly Score Distribution (Legit vs. Fraud)",
    xaxis_title="Anomaly Score (lower = more anomalous)",
    yaxis_title="Count",
    barmode="overlay",
    height=440,
)
fig.show()

print("💡 Legitimate transactions cluster around positive scores (normal)")
print("   Fraudulent transactions have more negative scores (anomalous)")
print("   The decision boundary at 0 separates the two groups")

In [ ]:
# Statistics
print("Anomaly Score Statistics:")
print(f"  Legit — Mean: {df[df['is_fraud']==0]['anomaly_score'].mean():.4f}, "
      f"Std: {df[df['is_fraud']==0]['anomaly_score'].std():.4f}")
print(f"  Fraud — Mean: {df[df['is_fraud']==1]['anomaly_score'].mean():.4f}, "
      f"Std: {df[df['is_fraud']==1]['anomaly_score'].std():.4f}")

# Separation quality
from scipy.stats import ttest_ind
t_stat, p_val = ttest_ind(
    df[df["is_fraud"]==0]["anomaly_score"],
    df[df["is_fraud"]==1]["anomaly_score"],
    equal_var=False
)
print(f"\nT-test (legit vs. fraud scores): t={t_stat:.2f}, p={p_val:.2e}")
print("  → Scores are significantly different (p < 0.001)")

---
## §7 · Contamination Parameter Tuning

The `contamination` parameter controls the **decision threshold**.
- Too low: miss fraud (low recall).
- Too high: too many false alarms (low precision).

In [ ]:
# Test different contamination values
contamination_values = [0.005, 0.01, 0.02, 0.03, 0.05, 0.08, 0.10]
contamination_results = []

for cont in contamination_values:
    # Train with this contamination
    iso = IsolationForest(n_estimators=100, contamination=cont, random_state=42, n_jobs=-1)
    iso.fit(X_scaled)
    
    # Predict
    preds = iso.predict(X_scaled)
    is_anom = (preds == -1).astype(int)
    
    # Metrics
    prec = precision_score(y, is_anom, zero_division=0)
    rec = recall_score(y, is_anom, zero_division=0)
    f1 = f1_score(y, is_anom, zero_division=0)
    n_flagged = is_anom.sum()
    
    contamination_results.append({
        "Contamination": cont,
        "Flagged": n_flagged,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
    })

df_cont = pd.DataFrame(contamination_results).round(4)
print("=" * 70)
print("CONTAMINATION PARAMETER TUNING")
print("=" * 70)
print(df_cont.to_string(index=False))

print(f"\n💡 Optimal contamination ≈ actual fraud rate ({N_FRAUD/N_TXN:.2%})")
print("   Too low → misses fraud (low recall)")
print("   Too high → too many false alarms (low precision)")

In [ ]:
# Visualize contamination tuning
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Precision & Recall vs. Contamination",
                                    "F1-Score vs. Contamination"])

fig.add_trace(go.Scatter(
    x=df_cont["Contamination"], y=df_cont["Precision"],
    mode="lines+markers", name="Precision",
    line=dict(color="#636EFA", width=2.5),
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_cont["Contamination"], y=df_cont["Recall"],
    mode="lines+markers", name="Recall",
    line=dict(color="#EF553B", width=2.5),
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_cont["Contamination"], y=df_cont["F1-Score"],
    mode="lines+markers", name="F1-Score",
    line=dict(color="#00CC96", width=3),
    marker=dict(size=10),
), row=1, col=2)

# Mark optimal
best_cont = df_cont.loc[df_cont["F1-Score"].idxmax(), "Contamination"]
fig.add_vline(x=best_cont, line_dash="dash", line_color="gray",
              annotation_text=f"Best: {best_cont:.3f}", row=1, col=1)
fig.add_vline(x=best_cont, line_dash="dash", line_color="gray", row=1, col=2)

fig.update_layout(height=420, legend=dict(orientation="h", y=1.12))
fig.update_xaxes(title_text="Contamination", row=1, col=1)
fig.update_xaxes(title_text="Contamination", row=1, col=2)
fig.show()

---
## §8 · t-SNE Visualization in 2D

t-SNE projects high-dimensional data into 2D while preserving local structure.
We color points by Isolation Forest prediction to see how anomalies cluster.

In [ ]:
# Run t-SNE on a subsample (for speed)
SUBSAMPLE = 10_000
idx_sub = np.random.choice(len(X_scaled), SUBSAMPLE, replace=False)
X_sub = X_scaled[idx_sub]
y_sub = y[idx_sub]
anomaly_sub = is_anomaly[idx_sub]

print(f"Running t-SNE on {SUBSAMPLE:,} samples …")
t0 = time.time()

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=500)
X_tsne = tsne.fit_transform(X_sub)

print(f"✓ Completed in {time.time() - t0:.2f}s")

# Create DataFrame for plotting
df_tsne = pd.DataFrame({
    "tsne_1": X_tsne[:, 0],
    "tsne_2": X_tsne[:, 1],
    "is_fraud": y_sub,
    "predicted_anomaly": anomaly_sub,
})

# Plot
fig = px.scatter(
    df_tsne, x="tsne_1", y="tsne_2",
    color="predicted_anomaly",
    color_discrete_map={0: "#00CC96", 1: "#EF553B"},
    title="t-SNE Visualization: Isolation Forest Predictions",
    labels={"predicted_anomaly": "Predicted Anomaly"},
    opacity=0.6,
)
fig.update_layout(height=500)
fig.show()

print("💡 Anomalies (red) should cluster in specific regions of the t-SNE space")
print("   If they're scattered randomly, the model isn't capturing structure")

In [ ]:
# ── t-SNE colored by TRUE fraud labels ───────────────────────────
fig = px.scatter(
    df_tsne, x="tsne_1", y="tsne_2",
    color="is_fraud",
    color_discrete_map={0: "#00CC96", 1: "#EF553B"},
    title="t-SNE Visualization: TRUE Fraud Labels",
    labels={"is_fraud": "True Fraud"},
    opacity=0.6,
)
fig.update_layout(height=500)
fig.show()

print("💡 Compare this with the Isolation Forest predictions above")
print("   Good model = red points overlap in both plots")

---
## §9 · Comparison with Labeled Data

In [ ]:
# Evaluate Isolation Forest against true labels
prec = precision_score(y, is_anomaly, zero_division=0)
rec = recall_score(y, is_anomaly, zero_division=0)
f1 = f1_score(y, is_anomaly, zero_division=0)

# Use anomaly scores for AUC (lower score = more anomalous, so negate)
roc_auc = roc_auc_score(y, -anomaly_scores)
pr_auc = average_precision_score(y, -anomaly_scores)

print("=" * 70)
print("ISOLATION FOREST EVALUATION (vs. True Labels)")
print("=" * 70)
print(f"\nPrecision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"PR-AUC   : {pr_auc:.4f}")

# Confusion matrix
cm = confusion_matrix(y, is_anomaly)
print(f"\nConfusion Matrix:")
print(f"  True Negatives : {cm[0, 0]:>8,}")
print(f"  False Positives: {cm[0, 1]:>8,}")
print(f"  False Negatives: {cm[1, 0]:>8,}")
print(f"  True Positives : {cm[1, 1]:>8,}")

In [ ]:
# Precision-Recall Curve
precision_curve, recall_curve, thresholds = precision_recall_curve(y, -anomaly_scores)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=recall_curve, y=precision_curve,
    mode="lines",
    name=f"PR Curve (AUC = {pr_auc:.4f})",
    line=dict(color="#636EFA", width=2.5),
))

fig.update_layout(
    title="Precision-Recall Curve (Isolation Forest)",
    xaxis_title="Recall",
    yaxis_title="Precision",
    height=440,
)
fig.show()

print("💡 PR-AUC is the gold standard for imbalanced anomaly detection")
print("   (ROC-AUC can be misleading when anomalies are rare)")

---
## §10 · Feature Importance (Which Features Drive Anomalies?)

Isolation Forest doesn't provide built-in feature importance, but we can
approximate it by comparing feature distributions between anomalies and normals.

In [ ]:
# Compare feature means: anomalies vs. normals
anomalies = df[df["predicted_anomaly"] == 1]
normals = df[df["predicted_anomaly"] == 0]

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Mean (Anomaly)": [anomalies[f].mean() for f in feature_names],
    "Mean (Normal)": [normals[f].mean() for f in feature_names],
})

# Compute absolute difference (normalized)
importance_df["Abs Difference"] = abs(importance_df["Mean (Anomaly)"] - importance_df["Mean (Normal)"])
importance_df["Relative Diff"] = importance_df["Abs Difference"] / (importance_df["Mean (Normal)"].abs() + 1e-8)
importance_df = importance_df.sort_values("Relative Diff", ascending=False)

print("=" * 70)
print("TOP 15 FEATURES DRIVING ANOMALIES")
print("=" * 70)
print(importance_df.head(15)[["Feature", "Mean (Anomaly)", "Mean (Normal)", "Relative Diff"]].round(4).to_string(index=False))

print("\n💡 These features have the largest difference between anomalies and normals")
print("   → Focus feature engineering and monitoring on these")

In [ ]:
# Visualize top 15
fig = px.bar(
    importance_df.head(15),
    x="Relative Diff",
    y="Feature",
    orientation="h",
    title="Feature Importance: Relative Difference (Anomaly vs. Normal)",
    color="Relative Diff",
    color_continuous_scale="Viridis",
)
fig.update_layout(height=500, yaxis=dict(autorange="reversed"))
fig.show()

---
## §11 · Visualization Dashboard

In [ ]:
# ── 11.1 Anomaly Score Histogram with Threshold ──────────────────
threshold = iso_forest.offset_  # the decision threshold

fig = go.Figure()

# All scores
fig.add_trace(go.Histogram(
    x=anomaly_scores, nbinsx=150,
    name="All Transactions",
    marker_color="#636EFA",
    opacity=0.6,
))

# Threshold line
fig.add_vline(x=threshold, line_dash="dash", line_color="red",
              annotation_text=f"Threshold: {threshold:.4f}")

fig.update_layout(
    title="Anomaly Score Distribution with Decision Threshold",
    xaxis_title="Anomaly Score (lower = more anomalous)",
    yaxis_title="Count",
    height=420,
)
fig.show()

In [ ]:
# ── 11.2 Business Impact Summary ─────────────────────────────────
avg_fraud_amount = df[df["is_fraud"]==1]["amount"].mean()
true_positives = cm[1, 1]
false_positives = cm[0, 1]
false_negatives = cm[1, 0]

fraud_caught_value = true_positives * avg_fraud_amount
review_cost = false_positives * 5.0  # £5 per review
fraud_missed = false_negatives * avg_fraud_amount

print("=" * 70)
print("BUSINESS IMPACT: Isolation Forest (200K transactions)")
print("=" * 70)
print(f"\nAvg fraud amount    : £{avg_fraud_amount:,.2f}")
print(f"True Positives      : {true_positives:,} (fraud caught)")
print(f"False Positives     : {false_positives:,} (legit flagged for review)")
print(f"False Negatives     : {false_negatives:,} (fraud missed)")
print(f"\nFraud value caught  : £{fraud_caught_value:,.0f}")
print(f"Review cost         : £{review_cost:,.0f}")
print(f"Fraud missed        : £{fraud_missed:,.0f}")
print(f"Net value           : £{fraud_caught_value - review_cost:,.0f}")

---
## §12 · Key Business Takeaway

> ### 🎯 Action Items for a Fintech Data Scientist
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Novel fraud types, no labeled data, first-pass anomaly filter |
> | **Contamination** | Set to expected fraud rate (e.g., 0.02 for 2%) |
> | **Features** | Use all engineered features (30+); IF handles high dimensions well |
> | **Evaluation** | PR-AUC (not accuracy) for imbalanced anomaly detection |
> | **Deployment** | Score in < 1 ms per transaction (suitable for real-time) |
>
> ### 💡 Revolut-Specific Guidelines
>
> 1. **Isolation Forest complements rules** (Module 11):
>    ```
>    Transaction → Rules (5 ms) → Flagged? → Block/Review
>                                ↓ Not flagged
>                         Isolation Forest (1 ms) → Anomaly? → Review
>                                ↓ Normal
>                         ML Model (50 ms) → Score → Decision
>    ```
>
> 2. **Retrain weekly** — fraud patterns evolve, and the model must adapt.
>
> 3. **Monitor the anomaly score distribution** — if it shifts, fraud patterns may be changing.
>
> 4. **Use feature importance** to guide feature engineering:
>    - Features with high "Relative Diff" are the most discriminative.
>    - Focus monitoring and engineering on these features.
>
> ### 📌 Isolation Forest Cheat Sheet
>
> ```python
> from sklearn.ensemble import IsolationForest
>
> iso = IsolationForest(
>     n_estimators=100,           # more trees = more stable, slower
>     contamination=0.02,         # expected anomaly rate
>     max_samples="auto",         # subsample size (256 default)
>     max_features=1.0,           # feature subsampling
>     random_state=42,
>     n_jobs=-1,                  # parallelize
> )
>
> iso.fit(X_train)                # train on "normal" data
> scores = iso.decision_function(X_test)  # anomaly scores
> preds = iso.predict(X_test)     # -1 = anomaly, 1 = normal
> ```
>
> ### 🔑 Key Takeaways
>
> 1. **Isolation Forest is unsupervised** — no labeled fraud data needed.
> 2. **Contamination must match the expected anomaly rate** — tune carefully.
> 3. **PR-AUC > ROC-AUC** for rare anomalies.
> 4. **t-SNE helps visualize** how anomalies cluster in high-dimensional space.
> 5. **Feature importance** (via mean comparison) guides monitoring and engineering.